In [0]:
process_name='pricing_data_silver_dimenstion'
Folder_name='daily_pricing'

In [0]:
%sql
create or replace table pricing_analytics.silver.state_dim1
select distinct STATE_NAME,
current_timestamp() as lake_house_inserted_date,
current_timestamp() as lake_house_updated_date
from pricing_analytics.silver.daily_pricing where LAKE_HOUSE_UPDATED_DATE>(select coalesce(max(process_table_date_time), date('1900-01-01')) from pricing_analytics.bronze.pipeline_control_logs where process_name='pricing_data_silver_dimenstion' and process_status='Completed')

In [0]:
%sql
create or replace table pricing_analytics.silver.state_dim2
select silverDim.STATE_NAME,
row_number() over(order by silverDim.STATE_NAME) as STATE_ID,
current_timestamp() as lake_house_inserted_date,
current_timestamp() as lake_house_updated_date
from pricing_analytics.silver.state_dim1 silverDim
left join pricing_analytics.gold.state_dim goldDim
on silverDim.STATE_NAME=goldDim.STATE_NAME
where goldDim.STATE_NAME is null

In [0]:
%sql
create or replace table pricing_analytics.silver.state_dim3
select 
STATE_NAME,
(STATE_ID+PREV_MAX_ID) as STATE_ID,
current_timestamp() as lake_house_inserted_date,
current_timestamp() as lake_house_updated_date
from pricing_analytics.silver.state_dim2 silverDim
cross join (select coalesce(max(STATE_ID), 0) as PREV_MAX_ID from pricing_analytics.gold.state_dim)

In [0]:
%sql
insert into pricing_analytics.gold.state_dim(
  select STATE_NAME,
  STATE_ID,
  current_timestamp() as lake_house_inserted_date,
  current_timestamp() as lake_house_updated_date
  from pricing_analytics.silver.state_dim3
)

In [0]:
%sql
create or replace table pricing_analytics.silver.market_dim1
select 
distinct  MARKET_NAME,
current_timestamp() as lake_house_inserted_date,
current_timestamp() as lake_house_updated_date
from pricing_analytics.silver.daily_pricing where LAKE_HOUSE_UPDATED_DATE>(select coalesce(max(process_table_date_time), date('1900-01-01')) from pricing_analytics.bronze.pipeline_control_logs where process_name='pricing_data_silver_dimenstion' and process_status='Completed')

In [0]:
%sql
create or replace table pricing_analytics.silver.market_dim2
select silverDim.MARKET_NAME,
row_number() over(order by silverDim.MARKET_NAME) as MARKET_ID,
current_timestamp() as lake_house_inserted_date,
current_timestamp() as lake_house_updated_date
from pricing_analytics.silver.market_dim1 silverDim
left join pricing_analytics.gold.market_dim goldDim
on silverDim.MARKET_NAME=goldDim.MARKET_NAME
where goldDim.MARKET_NAME is null

In [0]:
%sql
create or replace table pricing_analytics.silver.market_dim3
select 
MARKET_NAME,
(MARKET_ID+PREV_MAX_ID) as MARKET_ID,
current_timestamp() as lake_house_inserted_date,
current_timestamp() as lake_house_updated_date
from pricing_analytics.silver.market_dim2
cross join (select coalesce(max(MARKET_ID), 0) as PREV_MAX_ID from pricing_analytics.gold.market_dim)

In [0]:
%sql
insert into pricing_analytics.gold.market_dim (
  select MARKET_NAME,
  MARKET_ID,
  current_timestamp() as lake_house_inserted_date,
  current_timestamp() as lake_house_updated_date
  from pricing_analytics.silver.market_dim3
)

In [0]:
%sql
create or replace table pricing_analytics.silver.product_dim1
select distinct PRODUCT_NAME,
PRODUCTGROUP_NAME,
current_timestamp() as lake_house_inserted_date,
current_timestamp() as lake_house_updated_date
from pricing_analytics.silver.daily_pricing where LAKE_HOUSE_UPDATED_DATE>(select coalesce(max(process_table_date_time), date('1900-01-01')) from pricing_analytics.bronze.pipeline_control_logs where process_name='pricing_data_silver_dimenstion' and process_status='Completed')

In [0]:
%sql
create or replace table pricing_analytics.silver.product_dim2
select silverDim.PRODUCT_NAME,
silverDim.PRODUCTGROUP_NAME,
row_number() over(order by silverDim.PRODUCT_NAME, silverDim.PRODUCTGROUP_NAME) as PRODUCT_ID,
current_timestamp() as lake_house_inserted_date,
current_timestamp() as lake_house_updated_date
from pricing_analytics.silver.product_dim1 silverDim
left join pricing_analytics.gold.product_dim goldDim
on silverDim.PRODUCT_NAME=goldDim.PRODUCT_NAME
where goldDim.PRODUCT_NAME is null

In [0]:
%sql
create or replace table pricing_analytics.silver.product_dim3
select 
PRODUCT_NAME,
PRODUCTGROUP_NAME,
(PRODUCT_ID+PREV_MAX_ID) as PRODUCT_ID,
current_timestamp() as lake_house_inserted_date,
current_timestamp() as lake_house_updated_date
from pricing_analytics.silver.product_dim2
cross join (select coalesce(max(PRODUCT_ID), 0) as PREV_MAX_ID from pricing_analytics.gold.product_dim)

In [0]:
%sql
insert into pricing_analytics.gold.product_dim(
  select 
  PRODUCT_NAME,
  PRODUCTGROUP_NAME,
  PRODUCT_ID,
  current_timestamp() as lake_house_inserted_date,
  current_timestamp() as lake_house_updated_date
from pricing_analytics.silver.product_dim3
)

In [0]:
%sql
insert into pricing_analytics.bronze.pipeline_control_logs(
  select
  'pricing_data_silver_dimenstion',
  max(LAKE_HOUSE_UPDATED_DATE),
  'Completed',
  current_timestamp()
  from pricing_analytics.silver.daily_pricing
)